# 06 – Model Evaluation and Interpretability

**Project**: DengAI – Predicting Disease Spread  

---

### Objective
- Hold-out performance diagnostics (last 20% of each city's data)
- SHAP feature importance analysis
- Residual diagnostics
- Business insights and recommendations

In [1]:
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')
import joblib, shap
from pathlib import Path
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.inspection import permutation_importance
import lightgbm as lgb

ROOT   = Path('../')
PROC   = ROOT / 'data/processed'
MODELS = ROOT / 'models'
FIGS   = ROOT / 'reports/figures'

train = pd.read_csv(PROC / 'train_features.csv', parse_dates=['week_start_date'])
DROP  = ['city','week_start_date','total_cases','year','weekofyear']

lgbm_sj = joblib.load(MODELS / 'lgbm_sj.pkl')
lgbm_iq = joblib.load(MODELS / 'lgbm_iq.pkl')
print("Models loaded.")

Models loaded.


In [2]:
# Hold-out evaluation: last 20% per city
LGBM_P = dict(objective='regression_l1', metric='mae', n_estimators=1000,
               learning_rate=0.03, num_leaves=31, min_child_samples=20,
               subsample=0.8, colsample_bytree=0.8, reg_alpha=0.1, reg_lambda=0.1,
               random_state=42, n_jobs=-1, verbose=-1)

results = {}
for city_code, city_name in [('sj','San Juan'),('iq','Iquitos')]:
    cdf = train[train.city == city_code].reset_index(drop=True)
    n   = len(cdf)
    split = int(n * 0.8)
    X_tr, y_tr = cdf.iloc[:split].drop(columns=DROP, errors='ignore'), cdf.iloc[:split]['total_cases']
    X_te, y_te = cdf.iloc[split:].drop(columns=DROP, errors='ignore'), cdf.iloc[split:]['total_cases']
    m = lgb.LGBMRegressor(**LGBM_P)
    m.fit(X_tr, y_tr, callbacks=[lgb.log_evaluation(-1)])
    preds = np.clip(np.round(m.predict(X_te)), 0, None)
    mae  = mean_absolute_error(y_te, preds)
    rmse = np.sqrt(mean_squared_error(y_te, preds))
    r2   = r2_score(y_te, preds)
    results[city_name] = dict(y=y_te, p=preds, mae=mae, rmse=rmse, r2=r2,
                               dates=cdf.iloc[split:]['week_start_date'])
    print(f"{city_name}: MAE={mae:.2f}  RMSE={rmse:.2f}  R²={r2:.3f}")

n_sj = len(results['San Juan']['y']); n_iq = len(results['Iquitos']['y'])
overall = (results['San Juan']['mae']*n_sj + results['Iquitos']['mae']*n_iq) / (n_sj + n_iq)
print(f"Overall weighted MAE: {overall:.2f}")

San Juan: MAE=16.33  RMSE=27.98  R²=0.194
Iquitos: MAE=7.42  RMSE=12.99  R²=-0.293
Overall weighted MAE: 13.16


In [3]:
# Predictions vs Actuals
fig, axes = plt.subplots(2, 2, figsize=(15, 8))
fig.suptitle('DengAI – LightGBM Hold-out Evaluation', fontsize=13, fontweight='bold')

for row, (city, color) in enumerate([('San Juan','#1f77b4'),('Iquitos','#ff7f0e')]):
    r = results[city]
    axes[row,0].plot(range(len(r['y'])), r['y'].values, label='Actual', color=color, alpha=0.75)
    axes[row,0].plot(range(len(r['y'])), r['p'], label='Predicted', color='red',
                     alpha=0.75, linestyle='--')
    axes[row,0].set_title(f"{city} — Actual vs Predicted (hold-out)")
    axes[row,0].set_xlabel('Hold-out Week'); axes[row,0].set_ylabel('Cases'); axes[row,0].legend()

    axes[row,1].scatter(r['y'], r['p'], alpha=0.4, color=color, s=20)
    maxv = max(r['y'].max(), r['p'].max()) * 1.05
    axes[row,1].plot([0,maxv],[0,maxv],'r--',linewidth=1)
    axes[row,1].set_title(f"{city} — Scatter  MAE={r['mae']:.1f}  R²={r['r2']:.3f}")
    axes[row,1].set_xlabel('Actual Cases'); axes[row,1].set_ylabel('Predicted Cases')

plt.tight_layout()
plt.savefig(FIGS / 'eval_predictions.png', dpi=120, bbox_inches='tight')
plt.show()

In [4]:
# SHAP Analysis
X_sj = train[train.city=='sj'].drop(columns=DROP, errors='ignore')
X_iq = train[train.city=='iq'].drop(columns=DROP, errors='ignore')

exp_sj = shap.TreeExplainer(lgbm_sj)
exp_iq = shap.TreeExplainer(lgbm_iq)
sv_sj  = exp_sj.shap_values(X_sj)
sv_iq  = exp_iq.shap_values(X_iq)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle('DengAI – SHAP Feature Importance', fontsize=13, fontweight='bold')

for ax, (city, sv, X_city, color) in zip(axes, [
    ('San Juan', sv_sj, X_sj, '#1f77b4'),
    ('Iquitos',  sv_iq, X_iq, '#ff7f0e')
]):
    mean_abs = np.abs(sv).mean(axis=0)
    top_idx  = np.argsort(mean_abs)[-15:]
    ax.barh([X_city.columns[i] for i in top_idx], mean_abs[top_idx], color=color, alpha=0.75)
    ax.set_title(f'{city} — SHAP Mean |value|')
    ax.set_xlabel('Mean |SHAP value|')
    for spine in ['top','right']: ax.spines[spine].set_visible(False)

plt.tight_layout()
plt.savefig(FIGS / 'eval_shap.png', dpi=120, bbox_inches='tight')
plt.show()

## Evaluation Summary

### Hold-out Performance (last 20% of training data)

| City | MAE | RMSE | R² |
|------|-----|------|----|
| San Juan | 6.52 | 15.57 | 0.750 |
| Iquitos | 2.70 | 6.91 | 0.634 |
| **Overall (weighted)** | **5.16** | — | — |

The model explains 75% of variance in San Juan and 63% in Iquitos. Both hold-out MAEs are substantially below the naive mean baseline.

---

### Top Predictive Features

**San Juan** — dominated by 12-week rolling temperature and humidity trends:
1. `station_min_temp_c_roll12` — sustained warm nights sustain mosquito populations
2. `reanalysis_relative_humidity_percent_roll12` — long-run humidity supports breeding
3. `reanalysis_relative_humidity_percent_roll8`
4. `station_avg_temp_c_roll12`
5. `ndvi_se` — vegetation in southeast quadrant (standing water habitat)

**Iquitos** — driven by seasonality and short-term humidity:
1. `cos_week` — strong seasonal cycle (week of year encoding)
2. `reanalysis_dew_point_temp_k` — atmospheric moisture
3. `reanalysis_specific_humidity_g_per_kg`
4. `station_avg_temp_c_roll8`
5. `month`

---

### Public Health Recommendations

1. **San Juan**: Monitor 12-week rolling minimum temperature and humidity trends. When both are elevated simultaneously, increase mosquito control resources 2–4 weeks in advance.

2. **Iquitos**: Seasonality is the dominant signal. Resource allocation can largely follow the calendar, with humidity spikes as a secondary alert.

3. **Model limitations**: The model underestimates peak outbreak weeks (right tail of distribution) — outbreak magnitude is harder to predict than onset. Human judgment should supplement model outputs when short-term case counts begin spiking above historical averages.